# Gemini 파인튜닝 (이형 면접 코치)

## 실행 순서
1. `LLM용JSON.ipynb` 실행 → `gemini_output/gemini_train.jsonl` 생성
2. 이 노트북 순서대로 실행
3. 튜닝 완료 후 모델 ID를 `02_eval_pipeline.ipynb` / `03_feedback.ipynb` 에 반영

**베이스 모델:** `models/gemini-1.5-flash-001-tuning` (무료 튜닝 지원)  
**소요 시간:** 데이터 양에 따라 20분~1시간

In [1]:
%pip install google-genai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, time
from pathlib import Path
from dotenv import load_dotenv

BASE       = Path(r'C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)')
DATA_PATH  = BASE / '샘플영상' / '이형 유튜브 영상' / 'DB용' / 'gemini_output' / 'gemini_train.jsonl'

load_dotenv(BASE / '.env')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')

if not GOOGLE_API_KEY:
    raise ValueError(f'.env 파일에 GOOGLE_API_KEY가 없습니다: {BASE / ".env"}')

print('설정 완료')
print(f'학습 데이터 경로: {DATA_PATH}')
print(f'파일 존재: {DATA_PATH.exists()}')

설정 완료
학습 데이터 경로: C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)\샘플영상\이형 유튜브 영상\DB용\gemini_output\gemini_train.jsonl
파일 존재: True


---
## STEP 1 — 학습 데이터 로드 & 검증

In [3]:
# 데이터 로드
with open(DATA_PATH, encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]

print(f'총 샘플: {len(rows)}개')
print(f'\n첫 번째 샘플 구조:')
r = rows[0]
print(f'  keys: {list(r.keys())}')
print(f'  text_input 앞100자: {r["text_input"][:100]}')
print(f'  output 앞100자: {r["output"][:100]}')

# 형식 검증
errors = []
for i, row in enumerate(rows):
    if 'text_input' not in row or 'output' not in row:
        errors.append(f'행 {i}: text_input/output 키 없음')
    elif not row['text_input'] or not row['output']:
        errors.append(f'행 {i}: 빈 값')

if errors:
    print(f'\n⚠ 형식 오류 {len(errors)}개:')
    for e in errors[:5]:
        print(f'  {e}')
else:
    print(f'\n✅ 형식 검증 통과 — 모든 {len(rows)}개 샘플 정상')

총 샘플: 68개

첫 번째 샘플 구조:
  keys: ['text_input', 'output']
  text_input 앞100자: 당신은 대기업 인사담당자 출신의 직설적인 면접 코치 '이형'이다.
10년간 수천 명을 면접했고, 지원자 답변을 들으면 합격/불합격이 바로 보인다.
친근한 말투를 쓰되 평가는 냉정하
  output 앞100자: [이형의 팩폭 한줄평]
면접의 필살기 라인업을 정리해드리면 첫번째, 정규직 전환 경험이 있다.

[이형의 시선]
면접의 필살기 라인업을 정리해드리면 첫번째, 정규직 전환 경험이 있

✅ 형식 검증 통과 — 모든 68개 샘플 정상


---
## STEP 2 — Gemini 클라이언트 초기화

In [4]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

# 튜닝 가능한 모델 목록 확인
print('튜닝 가능한 모델:')
for m in client.models.list():
    if 'tuning' in m.name.lower() or 'tunedModel' in str(m.supported_actions or []):
        print(f'  {m.name}')

튜닝 가능한 모델:


---
## STEP 3 — 파인튜닝 시작

> **한 번만 실행하세요.** 실행하면 튜닝 작업이 서버에 제출됩니다.

In [7]:
# 학습 데이터를 TuningExample 리스트로 변환
training_examples = [
    types.TuningExample(
        text_input=row['text_input'],
        output=row['output']
    )
    for row in rows
]

print(f'학습 샘플 준비: {len(training_examples)}개')

# 파인튜닝 시작 (InlineFinetuningDataset 없이 examples 직접 전달)
tuning_job = client.tunings.tune(
    base_model='models/gemini-1.5-flash-001-tuning',
    training_dataset=types.TuningDataset(
        examples=training_examples          # ← 직접 전달
    ),
    config=types.CreateTuningJobConfig(
        epoch_count=3,
        learning_rate_multiplier=1.0,
        tuned_model_display_name='이형_면접코치_v1'
    )
)

print(f'\n튜닝 작업 시작됨!')
print(f'작업 이름: {tuning_job.name}')
print(f'상태: {tuning_job.state}')
print(f'\n이 이름을 메모해두세요: {tuning_job.name}')

학습 샘플 준비: 68개


ServerError: 501 UNIMPLEMENTED. {'error': {'code': 501, 'message': 'Operation is not implemented, or supported, or enabled.', 'status': 'UNIMPLEMENTED'}}

---
## STEP 4 — 진행 상황 모니터링

튜닝은 서버에서 비동기로 실행됩니다. 이 셀을 반복 실행해서 상태를 확인하세요.

In [ ]:
# 현재 상태 확인 (주기적으로 재실행)
job = client.tunings.get(name=tuning_job.name)

print(f'상태: {job.state}')
if hasattr(job, 'tuned_model') and job.tuned_model:
    print(f'튜닝된 모델 ID: {job.tuned_model.model}')
if hasattr(job, 'error') and job.error:
    print(f'에러: {job.error}')

STATE_MAP = {
    'JOB_STATE_PENDING':    '⏳ 대기 중',
    'JOB_STATE_RUNNING':    '🔄 학습 중 (20~60분 소요)',
    'JOB_STATE_SUCCEEDED':  '✅ 완료!',
    'JOB_STATE_FAILED':     '❌ 실패',
    'JOB_STATE_CANCELLED':  '취소됨',
}
print(STATE_MAP.get(str(job.state), str(job.state)))

In [ ]:
# 완료될 때까지 자동 대기 (선택 — 최대 2시간)
print('튜닝 완료 대기 중... (Ctrl+C로 중단 가능, 서버 작업은 계속 실행됨)')

for i in range(240):   # 2시간 = 240 * 30초
    job = client.tunings.get(name=tuning_job.name)
    state = str(job.state)
    if state == 'JOB_STATE_SUCCEEDED':
        print(f'\n✅ 튜닝 완료!')
        print(f'모델 ID: {job.tuned_model.model}')
        break
    elif state == 'JOB_STATE_FAILED':
        print(f'\n❌ 튜닝 실패: {job.error}')
        break
    else:
        elapsed = (i + 1) * 30
        print(f'  [{elapsed//60}분 {elapsed%60}초 경과] 상태: {state}', end='\r')
        time.sleep(30)
else:
    print('\n⏰ 2시간 초과 — STEP 4의 상태 확인 셀로 수동 확인하세요.')

---
## STEP 5 — 튜닝된 모델 테스트 & ID 저장

In [ ]:
# 완료 후 실행 — 모델 ID 확인
job = client.tunings.get(name=tuning_job.name)

if str(job.state) != 'JOB_STATE_SUCCEEDED':
    print(f'아직 완료 안 됨 — 상태: {job.state}')
else:
    TUNED_MODEL_ID = job.tuned_model.model
    print(f'튜닝된 모델 ID: {TUNED_MODEL_ID}')
    print()
    print('▼ 아래 ID를 02_eval_pipeline.ipynb와 03_feedback.ipynb의 MODEL 변수에 붙여넣으세요:')
    print(f'  MODEL = "{TUNED_MODEL_ID}"')
    
    # 모델 ID 파일로 저장
    model_id_path = BASE / 'tuned_model_id.txt'
    model_id_path.write_text(TUNED_MODEL_ID, encoding='utf-8')
    print(f'\nID 저장됨: {model_id_path}')

In [ ]:
# 튜닝된 모델로 테스트
test_q = '1분 자기소개를 해보세요.'
test_a = '저는 성실하고 책임감 있는 사람입니다. 열심히 하겠습니다.'

prompt = f"""면접 질문: {test_q}
지원자 답변: {test_a}

위 답변에 대해 이형 스타일로 피드백하라."""

resp = client.models.generate_content(
    model=TUNED_MODEL_ID,
    contents=prompt
)

print('=== 튜닝 모델 테스트 결과 ===')
print(resp.text)

---
## 튜닝 완료 후 할 일

1. **`02_eval_pipeline.ipynb`** — cell-2의 `MODEL` 변수를 튜닝 모델 ID로 교체
2. **`03_feedback.ipynb`** — cell-2의 `MODEL` 변수를 튜닝 모델 ID로 교체
3. `02_eval_pipeline.ipynb` 재실행 → 베이스 모델 vs 튜닝 모델 점수 비교

```python
# 변경 예시 (두 파일 모두)
MODEL = 'tunedModels/이형-면접코치-v1-abcd1234'   # ← 실제 ID로 교체
```